# Pathway-ranking neural network with top-pathway RankNet consistency

This notebook implements the complete ranking pipeline from scratch. It learns
\(f_\theta:\mathbb{R}^G\rightarrow\mathbb{R}^P\), but the pathway scores are
intermediate values: **their ordering is the learning target**.

For a main dataset and each derived sub-dataset, the same network is used. The
detached main ranking supplies pseudo-target pair relationships; binary cross-entropy
trains each sub-dataset to reproduce those relationships, focusing by default on
the top 15% of valid main pathways.

Default equations:

\[
x_g=-\log_{10}(p_g)\,\mathrm{logFC}_g,\quad
h_g=\operatorname{ReLU}(w_gx_g+b_g)
\]
\[
z_p={\sum_gM_{pg}A_gh_g\over\sum_gM_{pg}A_g},\quad
s_p=w_pz_p+b_p
\]
\[
S_{ij}\in\{-1,0,1\},\quad t_{ij}=(1+S_{ij})/2,\quad
\ell_{ij}=\operatorname{binary cross-entropy}(\gamma(s_i^{sub}-s_j^{sub}),t_{ij})
\]

The earlier prose in the specification also showed multiplication by raw
`logFC`; this notebook intentionally uses the full log-fold-change requested for
the final model.

Data mode is `precomputed_subdatasets`. Differential-expression recalculation is
deliberately outside the neural model; no statistical test is silently invented.


In [11]:
from __future__ import annotations

import copy
import math
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Mapping, Optional, Sequence, Union

import numpy as np
import pandas as pd
import torch
from torch import Tensor, nn
import torch.nn.functional as F

@dataclass
class Config:
    """All experiment settings; defaults match the requested algorithm."""
    seed: int = 42
    p_value_epsilon: float = 1e-300
    missing_gene_value: float = 0.0
    subdataset_mode: str = "precomputed_subdatasets"
    main_margin: float = 0.1
    gamma: float = 1.0
    top_pathway_fraction: float = 0.15
    top_pathway_count: Optional[int] = None
    sub_tie_margin: float = 0.1
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    gradient_clip_norm: float = 5.0
    maximum_epochs: int = 300
    evaluation_top_k: tuple[int, ...] = (10, 20, 30)

CONFIG = Config()
random.seed(CONFIG.seed); np.random.seed(CONFIG.seed); torch.manual_seed(CONFIG.seed)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)


device: cpu


## Data, gene rank, vocabulary, and dataset grouping

The normalized long table has `dataset_id`, `sub_id`, `sample_type`, `gene_id`,
`p_value`, and `log_fold_change`. `load_deg_csv` also adapts the repository's
limma-style files (`P.Value`, `logFC`, gene symbols in the first column).
Splitting is performed only on unique `dataset_id` values, so related
sub-datasets cannot leak across partitions.

In [2]:
def compute_gene_rank(
    p_value: Tensor, log_fold_change: Tensor, epsilon: float = 1e-300,
) -> Tensor:
    """Compute significance weighted by log-fold-change.

    Algorithm mapping:
        ALGORITHM STEP 1 — Gene ranking input. Each gene is represented by
        x_g = -log10(p_g) * logFC_g. The sign represents expression direction,
        while the magnitude preserves effect size.
    """
    # if epsilon <= 0 or epsilon > 1:
    #     raise ValueError("epsilon must be in (0, 1]")
    # if not torch.isfinite(p_value).all() or (p_value < 0).any():
    #     raise ValueError("p-values must be finite and non-negative")
    # if not torch.isfinite(log_fold_change).all():
    #     raise ValueError("log-fold-changes must be finite")
    p = p_value.clamp(epsilon, 1.0)
    return -torch.log10(p) * log_fold_change


@dataclass(frozen=True)
class GeneVocabulary:
    """Fixed global ordering used by every main/sub-dataset and checkpoint."""
    genes: tuple[str, ...]

    # def __post_init__(self):
    #     if len(set(self.genes)) != len(self.genes):
    #         raise ValueError("gene vocabulary contains duplicates")

    @classmethod
    def from_genes(cls, genes: Iterable[str]):
        return cls(tuple(sorted({str(g) for g in genes})))

    @property
    def index(self):
        return {g: i for i, g in enumerate(self.genes)}

    def __len__(self):
        return len(self.genes)


def align_gene_statistics(
    gene_ids: Sequence[str], p_values: Sequence[float],
    log_fold_changes: Sequence[float], vocabulary: GeneVocabulary,
    config: Config = CONFIG,
) -> tuple[Tensor, Tensor]:
    """Align arbitrary gene rows to the fixed G-dimensional vocabulary.

    Algorithm mapping:
        Missing genes receive zero and availability 0. Unknown genes are
        ignored with a warning. Duplicate rows are rejected because silently
        choosing one would change the biological input.
    """
    if not (len(gene_ids) == len(p_values) == len(log_fold_changes)):
        raise ValueError("gene_ids, p_values, and log_fold_changes must align")
    if len(set(map(str, gene_ids))) != len(gene_ids):
        raise ValueError("duplicate gene rows are not allowed")
    values = torch.full((len(vocabulary),), float(config.missing_gene_value))
    available = torch.zeros(len(vocabulary), dtype=torch.bool)
    idx = vocabulary.index
    unknown = []
    for gene, p, fc in zip(gene_ids, p_values, log_fold_changes):
        gene = str(gene)
        if gene not in idx:
            unknown.append(gene); continue
        rank = compute_gene_rank(
            torch.tensor(float(p)), torch.tensor(float(fc)),
            config.p_value_epsilon,
        )
        values[idx[gene]], available[idx[gene]] = rank, True
    if unknown:
        warnings.warn(f"Ignored {len(unknown)} unknown genes; first: {unknown[:3]}")
    return values, available


def build_gene_input_vector(records, vocabulary, config=CONFIG):
    """Public record-oriented wrapper for fixed-vector construction.

    Algorithm mapping:
        Converts one main or sub-dataset into gene_values [G] and
        gene_available [G] using the global ordering.
    """
    frame = pd.DataFrame(records)
    return align_gene_statistics(
        frame.gene_id.astype(str).tolist(), frame.p_value.tolist(),
        frame.log_fold_change.tolist(), vocabulary, config,
    )


def load_deg_csv(path: Union[str, Path], dataset_id: Optional[str] = None,
                 sub_id: str = "main", sample_type: str = "main") -> pd.DataFrame:
    """Load normalized data or adapt the repository's limma DEG CSV schema."""
    df = pd.read_csv(path)
    if {"gene_id", "p_value", "log_fold_change"}.issubset(df.columns):
        out = df.copy()
    elif {"P.Value", "logFC"}.issubset(df.columns):
        gene_col = df.columns[0]
        out = df.rename(columns={gene_col: "gene_id", "P.Value": "p_value",
                                 "logFC": "log_fold_change"})
    else:
        raise ValueError(f"Unrecognized DEG schema in {path}")
    out["dataset_id"] = out.get("dataset_id", dataset_id or Path(path).stem)
    out["sub_id"] = out.get("sub_id", sub_id)
    out["sample_type"] = out.get("sample_type", sample_type)
    return out[["dataset_id", "sub_id", "sample_type", "gene_id",
                "p_value", "log_fold_change"]]


@dataclass
class DatasetGroup:
    dataset_id: str
    main_batch: tuple[Tensor, Tensor]
    sub_batches: list[tuple[Tensor, Tensor]]


# def group_main_and_subdatasets(table: pd.DataFrame, vocabulary: GeneVocabulary,
#                                config: Config = CONFIG) -> list[DatasetGroup]:
#     """Group one main dataset with all its sub-datasets.

#     Algorithm mapping:
#         This materializes the training unit required by ALGORITHM STEP 12;
#         subdataset counts may differ among main datasets.
#     """
#     required = {"dataset_id","sub_id","sample_type","gene_id","p_value","log_fold_change"}
#     missing = required - set(table.columns)
#     if missing: raise ValueError(f"missing columns: {sorted(missing)}")
#     groups = []
#     for dataset_id, ds in table.groupby("dataset_id", sort=True):
#         mains = ds[ds.sample_type.eq("main")]
#         if mains.empty: raise ValueError(f"{dataset_id} has no main rows")
#         # A main may repeat sub_id='main'; combine all main rows as one input.
#         main = build_gene_input_vector(mains, vocabulary, config)
#         subs = [build_gene_input_vector(part, vocabulary, config)
#                 for _, part in ds[ds.sample_type.eq("sub")].groupby("sub_id", sort=True)]
#         if not subs: warnings.warn(f"{dataset_id} has no sub-datasets")
#         groups.append(DatasetGroup(str(dataset_id), main, subs))
#     return groups


def build_masked_subdataset_groups(
    main_tables: Mapping[str, pd.DataFrame], vocabulary: GeneVocabulary,
    subdatasets_per_dataset: int = 3,
    retained_gene_fraction: Union[float, Sequence[float]] = 0.70,
    seed: int = 42, config: Config = CONFIG,
) -> list[DatasetGroup]:
    """Build runnable main/sub groups from existing main DEG CSV statistics.

    For every requested retention fraction, create subdatasets_per_dataset
    sub-inputs that retain a random fraction of genes from the main input.
    This simulates missing genes; it is not a replacement for recomputing DEG
    statistics from independently resampled expression-level observations.
    """
    if subdatasets_per_dataset < 1:
        raise ValueError("subdatasets_per_dataset must be at least 1")
    if (isinstance(retained_gene_fraction, Sequence)
            and not isinstance(retained_gene_fraction, (str, bytes))):
        retained_gene_fractions = tuple(map(float, retained_gene_fraction))
    else:
        retained_gene_fractions = (float(retained_gene_fraction),)
    if not retained_gene_fractions:
        raise ValueError("retained_gene_fraction must not be empty")
    if any(not 0 < fraction <= 1 for fraction in retained_gene_fractions):
        raise ValueError("every retained_gene_fraction must be in (0, 1]")
    generator = torch.Generator().manual_seed(seed)
    groups = []
    for dataset_id, table in sorted(main_tables.items()):
        main_values, main_available = build_gene_input_vector(
            table, vocabulary, config)
        available_indices = main_available.nonzero().flatten()
        if available_indices.numel() == 0:
            raise ValueError(f"{dataset_id} has no genes in the vocabulary")
        sub_batches = []
        for retained_fraction in retained_gene_fractions:
            keep_count = max(
                1, math.ceil(retained_fraction * available_indices.numel()))
            for _ in range(subdatasets_per_dataset):
                chosen = available_indices[
                    torch.randperm(
                        available_indices.numel(), generator=generator
                    )[:keep_count]
                ]
                sub_available = torch.zeros_like(main_available)
                sub_available[chosen] = True
                sub_values = torch.full_like(
                    main_values, float(config.missing_gene_value))
                sub_values[chosen] = main_values[chosen]
                sub_batches.append((sub_values, sub_available))
        groups.append(DatasetGroup(
            str(dataset_id), (main_values, main_available), sub_batches))
    return groups


def split_dataset_ids(table: pd.DataFrame, fractions=(0.7, 0.15, 0.15), seed=42):
    """Split only unique dataset IDs, preventing main/sub leakage."""
    if not math.isclose(sum(fractions), 1.0): raise ValueError("fractions must sum to 1")
    ids = sorted(map(str, table.dataset_id.unique()))
    random.Random(seed).shuffle(ids)
    n, n_train = len(ids), int(len(ids) * fractions[0])
    n_val = int(len(ids) * fractions[1])
    return ids[:n_train], ids[n_train:n_train+n_val], ids[n_train+n_val:]


## KEGG pathway definitions and biological mask

CSV (`pathway_id,gene_id`) and GMT are supported. A gene can belong to many
pathways. Empty pathways and pathways with no vocabulary genes are rejected,
because they can never create a valid ranking score.

In [12]:
def load_pathways(path: Union[str, Path]) -> dict[str, set[str]]:
    """Load CSV or GMT pathway membership.

    Algorithm mapping:
        Reads the biological sets used to construct M[p,g].
    """
    path = Path(path)
    pathways: dict[str, set[str]] = {}
    if path.suffix.lower() == ".gmt":
        with path.open(encoding="utf-8") as f:
            for line in f:
                fields = line.rstrip("\n").split("\t")
                if len(fields) < 3: raise ValueError("GMT rows need name, description, genes")
                pathways.setdefault(fields[0], set()).update(filter(None, fields[2:]))
    elif path.suffix.lower() == ".csv":
        frame = pd.read_csv(path)
        if not {"pathway_id","gene_id"}.issubset(frame.columns):
            raise ValueError("pathway CSV needs pathway_id,gene_id")
        for name, part in frame.groupby("pathway_id"):
            pathways[str(name)] = set(part.gene_id.astype(str))
    else:
        raise ValueError("pathway file must be .gmt or .csv")
    if not pathways or any(not genes for genes in pathways.values()):
        raise ValueError("pathway definitions cannot be empty")
    return pathways


def build_pathway_gene_mask(pathways: Mapping[str, Iterable[str]],
                            vocabulary: GeneVocabulary):
    """Build dense binary M [P,G]; use .to_sparse() for very large masks.

    Algorithm mapping:
        ALGORITHM STEP 2 — Pathway-gene mask.
        M[p,g] = 1 means that gene g belongs to pathway p. The mask ensures
        pathway p is scored using only biologically assigned genes.
    """
    names, index = tuple(pathways), vocabulary.index
    mask = torch.zeros((len(names), len(vocabulary)), dtype=torch.float32)
    unknown = set()
    for p, name in enumerate(names):
        for gene in pathways[name]:
            if gene in index: mask[p, index[gene]] = 1
            else: unknown.add(str(gene))
    if unknown: warnings.warn(f"{len(unknown)} pathway genes absent from vocabulary")
    return names, mask


# def validate_pathway_gene_mask(mask: Tensor, pathway_names, vocabulary):
#     """Validate P x G binary membership and return per-pathway gene counts."""
#     if mask.shape != (len(pathway_names), len(vocabulary)):
#         raise ValueError("mask shape must be P x G")
#     if not torch.all((mask == 0) | (mask == 1)):
#         raise ValueError("pathway mask must be binary")
#     counts = mask.sum(1)
#     if (counts == 0).any():
#         bad = [pathway_names[i] for i in (counts == 0).nonzero().flatten().tolist()]
#         raise ValueError(f"pathways have no vocabulary genes: {bad[:5]}")
#     return counts


## Network: gene encoder → pathway average pooling → pathway scores

In [13]:
class GeneHiddenEncoder(nn.Module):
    """One trainable elementwise affine transform per gene.

    Algorithm mapping:
        ALGORITHM STEP 3 — h = ReLU(W1 x + b1), while retaining one hidden
        representation per gene for biological pathway pooling.
    """
    def __init__(self, gene_count: int):
        super().__init__()
        self.gene_weight = nn.Parameter(torch.ones(gene_count))
        self.gene_bias = nn.Parameter(torch.zeros(gene_count))

    def forward(self, x):
        return F.relu(self.gene_weight * x + self.gene_bias)


class PathwayAveragePool(nn.Module):
    """Vectorized available-gene mean pooling from [B,G] to [B,P].

    Algorithm mapping:
        ALGORITHM STEP 4 — For every pathway, sum available hidden member
        genes and divide by the available member count to obtain z_p.
    """
    def __init__(self, pathway_gene_mask: Tensor):
        super().__init__()
        self.register_buffer("mask", pathway_gene_mask.float())

    def forward(self, hidden_gene_values, gene_available):
        available = gene_available.float()
        numerator = (hidden_gene_values * available) @ self.mask.T
        counts = available @ self.mask.T
        valid = counts > 0
        pooled = torch.where(valid, numerator / counts.clamp_min(1), torch.zeros_like(numerator))
        return pooled, valid, counts


class PathwayScoreLayer(nn.Module):
    """Interpretable elementwise pathway scorer.

    Algorithm mapping:
        ALGORITHM STEP 5 — Each z_p becomes s_p=w_p*z_p+b_p; ordering matters,
        not absolute score magnitude.
    """
    def __init__(self, pathway_count):
        super().__init__()
        self.pathway_weight = nn.Parameter(torch.ones(pathway_count))
        self.pathway_bias = nn.Parameter(torch.zeros(pathway_count))

    def forward(self, z):
        return self.pathway_weight * z + self.pathway_bias


@dataclass
class PathwayNetworkOutput:
    gene_hidden: Tensor
    pathway_representations: Tensor
    pathway_scores: Tensor
    valid_pathway_mask: Tensor
    available_gene_counts: Tensor


class PathwayRankingNetwork(nn.Module):
    """Shared f_theta: R^G -> R^P used by main and every sub-dataset."""
    def __init__(self, gene_count, pathway_gene_mask, config=CONFIG):
        super().__init__()
        self.encoder = GeneHiddenEncoder(gene_count)
        self.pool = PathwayAveragePool(pathway_gene_mask)
        self.scorer = PathwayScoreLayer(pathway_gene_mask.shape[0])

    def forward(self, gene_values, gene_available):
        """Run algorithm steps 3–5 and return every interpretable stage."""
        h = self.encoder(gene_values)
        z, valid, counts = self.pool(h, gene_available)
        return PathwayNetworkOutput(h, z, self.scorer(z), valid, counts)


## Top-pathway pair selection and RankNet loss

In [14]:
def select_top_pathways(main_scores, main_valid_mask, top_fraction=0.15, top_count=None):
    """Select detached main-ranking leaders only.

    Algorithm mapping:
        ALGORITHM STEP 7 — Focus on top pathways. K is at least two when two
        valid pathways exist; invalid pathways are never selected.
    """
    scores, valid = main_scores.detach().flatten(), main_valid_mask.bool().flatten()
    valid_idx = valid.nonzero().flatten()
    if valid_idx.numel() < 2: return valid_idx
    k = top_count if top_count is not None else math.ceil(top_fraction * valid_idx.numel())
    k = min(valid_idx.numel(), max(2, int(k)))
    return valid_idx[torch.topk(scores[valid_idx], k).indices]


def build_pathway_pairs(selected_indices):
    """Construct every ordered, non-self pair among selected top pathways.

    Algorithm mapping:
        ALGORITHM STEP 8 — Includes both (i,j) and (j,i), never i == j.
    """
    selected = selected_indices.flatten()
    if selected.numel() < 2:
        empty = selected.new_empty(0)
        return empty, empty
    i = selected[:, None].expand(-1, len(selected)).reshape(-1)
    j = selected[None, :].expand(len(selected), -1).reshape(-1)
    keep = i != j
    return i[keep], j[keep]


def compute_main_ranking_targets(main_scores, pair_i, pair_j, margin=0.1):
    """Convert detached main differences into ternary RankNet targets.

    Algorithm mapping:
        ALGORITHM STEP 9 — delta_main receives S_ij in {-1,0,+1}; then
        target_ij=(1+S_ij)/2 creates {0,.5,1} pseudo-gold targets.
    """
    delta = (main_scores.detach()[pair_i] - main_scores.detach()[pair_j]).detach()
    relation = torch.where(delta > margin, 1., torch.where(delta < -margin, -1., 0.)).detach()
    return delta, relation, (0.5 * (1 + relation)).detach()


def compute_sub_pair_probability(sub_scores, pair_i, pair_j, gamma=1.0):
    """Compute the RankNet probability P_ij directly.

    Algorithm mapping:
        ALGORITHM STEP 10 — P_ij = sigmoid(gamma * (s_i - s_j)) estimates
        the probability that sub-dataset pathway i ranks above pathway j.
    """
    score_difference = sub_scores[pair_i] - sub_scores[pair_j]
    return torch.sigmoid(gamma * score_difference)


def compute_pairwise_accuracy(probabilities, targets):
    """Decisive-pair accuracy using P_ij > 0.5 as i-above-j."""
    decisive = targets != 0.5
    if not decisive.any(): return probabilities.new_tensor(0.)
    return ((probabilities[decisive] > 0.5) ==
            (targets[decisive] == 1)).float().mean()


def compute_tie_aware_agreement(probabilities, targets, gamma=1.0, tie_margin=0.1):
    """Tie-aware agreement expressed directly in probability space."""
    upper = torch.sigmoid(probabilities.new_tensor(gamma * tie_margin))
    lower = 1 - upper
    good = torch.where(targets == 1, probabilities > upper,
           torch.where(targets == 0, probabilities < lower,
                       (probabilities >= lower) & (probabilities <= upper)))
    return good.float().mean() if good.numel() else probabilities.new_tensor(0.)


@dataclass
class RankNetLossOutput:
    loss: Tensor
    number_of_pairs: int
    number_of_top_pathways: int
    positive_pairs: int
    negative_pairs: int
    tied_pairs: int
    pairwise_accuracy: Tensor
    pair_targets: Tensor
    pair_probabilities: Tensor
    pair_i: Tensor
    pair_j: Tensor


def top_pathway_ranknet_loss(main_scores, sub_scores, main_valid_mask, sub_valid_mask,
                             margin=0.1, gamma=1.0, top_fraction=0.15, top_count=None,
                             selected_indices=None):
    """Compute top-focused RankNet consistency loss.

    Algorithm mapping:
        ALGORITHM STEP 11 — Binary cross-entropy compares main-derived probability
        targets with sub pair predictions. Invalid pathways are excluded.
    """
    # joint_valid = main_valid_mask.bool() & sub_valid_mask.bool()
    # valid_idx = joint_valid.nonzero().flatten()
    # top = select_top_pathways(main_scores, main_valid_mask, top_fraction, top_count) \
    #     if selected_indices is None else selected_indices
    # top = top[joint_valid[top]] if top.numel() else top
    i, j = build_pathway_pairs(selected_indices)
    if i.numel() == 0:
        zero = sub_scores.sum() * 0
        empty = sub_scores.new_empty(0)
        return RankNetLossOutput(zero, 0, len(selected_indices), 0, 0, 0, zero.detach(),
                                 empty, empty, i, j)
    _, relation, target = compute_main_ranking_targets(main_scores, i, j, margin)
    probabilities = compute_sub_pair_probability(sub_scores, i, j, gamma)
    losses = F.binary_cross_entropy(probabilities, target, reduction="none")
    loss = losses.mean()
    return RankNetLossOutput(
        loss, len(i), len(selected_indices), int((relation > 0).sum()), int((relation < 0).sum()),
        int((relation == 0).sum()), compute_pairwise_accuracy(probabilities, target),
        target, probabilities, i, j,
    )


## Multiple sub-datasets, metrics, and training

The detached main ranking uses shared stop-gradient. Its pseudo-ranking can move because the same model is being updated.

In [ ]:
@dataclass
class PairSample:
    """One iteration sample: a full dataset paired with one of its sub-datasets."""
    dataset_id: str
    main_batch: tuple[Tensor, Tensor]
    sub_batch: tuple[Tensor, Tensor]


@dataclass
class SampleLossOutput:
    loss: Tensor
    main_output: PathwayNetworkOutput
    sub_output: PathwayNetworkOutput
    rank_output: RankNetLossOutput


def build_pair_samples(groups, subdatasets_per_dataset=None, seed=42, shuffle=True):
    """Create one [full dataset, related sub-dataset] row per iteration.

    Algorithm mapping:
        With D full datasets and K selected sub-datasets per full dataset,
        this creates D*K samples. Selection is random without replacement
        within each dataset, and never pairs a sub-dataset with another main.
    """
    rng = random.Random(seed)
    samples = []
    for group in groups:
        available = len(group.sub_batches)
        count = available if subdatasets_per_dataset is None else subdatasets_per_dataset
        if count < 1:
            raise ValueError("subdatasets_per_dataset must be at least 1")
        if count > available:
            raise ValueError(
                f"{group.dataset_id} has {available} sub-datasets, cannot select {count}")
        selected = rng.sample(group.sub_batches, count)
        samples.extend(PairSample(group.dataset_id, group.main_batch, sub)
                       for sub in selected)
    if shuffle:
        rng.shuffle(samples)
    return samples


def samples_to_input_matrix(samples):
    """Materialize Samples as gene values and availability [N,2,G] tensors."""
    if not samples:
        raise ValueError("at least one pair sample is required")
    values = torch.stack([
        torch.stack((sample.main_batch[0], sample.sub_batch[0]))
        for sample in samples
    ])
    available = torch.stack([
        torch.stack((sample.main_batch[1], sample.sub_batch[1]))
        for sample in samples
    ])
    return values, available


def compute_sample_consistency_loss(model, sample, config=CONFIG):
    """Run the full and sub-dataset separately and calculate one sample loss.

    Algorithm mapping:
        For sample [[X_main], [X_sub]], call the shared model first on X_main
        to obtain s_main and separately on X_sub to obtain s_sub. The detached
        s_main top ranking supplies targets for the RankNet loss on s_sub.
    """
    device = next(model.parameters()).device
    main_x, main_available = (
        tensor.unsqueeze(0).to(device) for tensor in sample.main_batch)
    sub_x, sub_available = (
        tensor.unsqueeze(0).to(device) for tensor in sample.sub_batch)

    # Two deliberately separate forward calls for this sample.
    main_output = model(main_x, main_available)
    sub_output = model(sub_x, sub_available)

    main_scores = main_output.pathway_scores[0].detach()
    main_valid = main_output.valid_pathway_mask[0]
    top = select_top_pathways(
        main_scores, main_valid,
        config.top_pathway_fraction, config.top_pathway_count,
    )
    rank_output = top_pathway_ranknet_loss(
        main_scores, sub_output.pathway_scores[0],
        main_valid, sub_output.valid_pathway_mask[0],
        config.main_margin, config.gamma,
        config.top_pathway_fraction, config.top_pathway_count, top,
    )
    return SampleLossOutput(
        rank_output.loss, main_output, sub_output, rank_output)


def train_one_sample(model, sample, optimizer, config=CONFIG):
    """Perform one optimizer update for one [full, sub] sample row."""
    model.train()
    optimizer.zero_grad(set_to_none=True)
    result = compute_sample_consistency_loss(model, sample, config)
    result.loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), config.gradient_clip_norm)
    optimizer.step()
    return float(result.loss.detach())


def train_one_epoch(model, samples, optimizer, config=CONFIG):
    """Shuffle and train on every [full, sub] sample exactly once."""
    ordered = list(samples)
    random.shuffle(ordered)
    return float(np.mean([
        train_one_sample(model, sample, optimizer, config) for sample in ordered
    ]))


@torch.no_grad()
def validate_one_epoch(model, samples, config=CONFIG):
    """Evaluate mean RankNet loss over independent [full, sub] samples."""
    model.eval()
    return float(np.mean([
        compute_sample_consistency_loss(model, sample, config).loss.item()
        for sample in samples
    ]))


@torch.no_grad()
def calculate_recall(main_output, sub_output, top_k=20):
    def top_pathway_set(output, output_name):
        scores = output.pathway_scores[0]
        valid_indices = output.valid_pathway_mask[0].bool().nonzero().flatten()
        if valid_indices.numel() < top_k:
            raise ValueError(
                f"{output_name} has only {valid_indices.numel()} valid pathways; "
                f"{top_k} are required")
        top_indices = valid_indices[
            torch.topk(scores[valid_indices], k=top_k).indices
        ]
        return set(top_indices.tolist())

    main_top = top_pathway_set(main_output, "main output")
    sub_top = top_pathway_set(sub_output, "sub output")
    return len(main_top & sub_top) / float(top_k)


@torch.no_grad()
def calculate_discrimination(
    own_main_output, other_main_outputs, sub_output, top_k=20,
):
    """Return 1 when own recall strictly exceeds every wrong recall.

    other_main_outputs may be a mapping or a sequence. A tie with any wrong
    main is a failure, following the article's strict discrimination rule.
    """
    if isinstance(other_main_outputs, Mapping):
        wrong_outputs = list(other_main_outputs.values())
    else:
        wrong_outputs = list(other_main_outputs)
    if not wrong_outputs:
        raise ValueError("at least one wrong-condition main output is required")

    own_recall = calculate_recall(own_main_output, sub_output, top_k)
    wrong_recalls = [
        calculate_recall(wrong_output, sub_output, top_k)
        for wrong_output in wrong_outputs
    ]
    return float(all(own_recall > wrong_recall for wrong_recall in wrong_recalls))


@torch.no_grad()
def test_model(
    model, test_samples, pathway_names, dataset_names=None,
    condition_by_dataset=None, config=CONFIG,
):
    """Evaluate every test sample and return loss, recall, and discrimination.

    Each history row represents one [main, sub] sample. When condition labels
    are omitted, every distinct dataset ID is treated as a distinct condition.
    """
    if not test_samples:
        raise ValueError("at least one test sample is required")
    names = tuple(map(str, pathway_names))
    name_by_id = dataset_names or {}
    condition_by_id = condition_by_dataset or {}
    evaluated_samples = []
    main_outputs = {}
    main_top_pathways = {}
    model.eval()

    for sample in test_samples:
        result = compute_sample_consistency_loss(model, sample, config)
        dataset_id = str(sample.dataset_id)
        evaluated_samples.append((sample, result))
        if dataset_id not in main_outputs:
            main_outputs[dataset_id] = result.main_output
            scores = result.main_output.pathway_scores[0]
            valid_indices = (
                result.main_output.valid_pathway_mask[0].bool().nonzero().flatten()
            )
            if len(names) != scores.numel():
                raise ValueError(
                    "pathway_names must align with the model pathway scores")
            if valid_indices.numel() < 20:
                raise ValueError(
                    f"{dataset_id} has only {valid_indices.numel()} valid pathways; "
                    "20 are required")
            top_indices = valid_indices[
                torch.topk(scores[valid_indices], k=20).indices
            ]
            main_top_pathways[dataset_id] = [
                names[i] for i in top_indices.tolist()
            ]

    rows = []
    subdataset_counts = {}
    for sample, result in evaluated_samples:
        dataset_id = str(sample.dataset_id)
        subdataset_counts[dataset_id] = subdataset_counts.get(dataset_id, 0) + 1
        own_condition = str(condition_by_id.get(dataset_id, dataset_id))
        wrong_main_outputs = {
            other_id: output
            for other_id, output in main_outputs.items()
            if (other_id != dataset_id and
                str(condition_by_id.get(other_id, other_id)) != own_condition)
        }
        recall = calculate_recall(
            result.main_output, result.sub_output, top_k=20)
        discrimination = calculate_discrimination(
            result.main_output, wrong_main_outputs, result.sub_output, top_k=20)
        main_gene_count = int(sample.main_batch[1].bool().sum())
        sub_gene_count = int(sample.sub_batch[1].bool().sum())
        if main_gene_count == 0:
            raise ValueError(f"{dataset_id} has no available main-dataset genes")
        rows.append({
            "test_loss": float(result.loss),
            "recall": recall,
            "discrimination": discrimination,
            "dataset_id": dataset_id,
            "dataset_name": str(name_by_id.get(dataset_id, dataset_id)),
            "subdataset_index": subdataset_counts[dataset_id],
            "retained_gene_fraction": round(
                sub_gene_count / main_gene_count, 4),
            "top_20_pathways": main_top_pathways[dataset_id],
        })

    history = pd.DataFrame(rows)
    return history


def train_model(model, train_samples, validation_samples, config=CONFIG):
    """Train sample by sample with AdamW for exactly maximum_epochs."""
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )
    best, history, best_state = math.inf, [], None
    for epoch in range(config.maximum_epochs):
        train_loss = train_one_epoch(model, train_samples, optimizer, config)
        val_loss = validate_one_epoch(
            model, validation_samples or train_samples, config)
        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "validation_loss": val_loss,
        })
        if val_loss < best:
            best = val_loss
            best_state = copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return pd.DataFrame(history), optimizer, {"validation_loss": best}


In [20]:
from types import SimpleNamespace

# Create example outputs
# -----------------------------------

main_output = SimpleNamespace(
    pathway_scores=[
        torch.tensor([
            0.95, 0.10, 0.85, 0.30, 0.75,
            0.20, 0.90, 0.40, 0.65, 0.55,
            0.80, 0.15, 0.70, 0.35, 0.60,
            0.45, 0.50, 0.25, 0.88, 0.78,
            0.12, 0.92, 0.68, 0.33, 0.58,
            0.73, 0.42, 0.82, 0.18, 0.63
        ])
    ],
    valid_pathway_mask=[
        torch.ones(30, dtype=torch.bool)
    ]
)


sub_output = SimpleNamespace(
    pathway_scores=[
        torch.tensor([
            0.90, 0.20, 0.80, 0.35, 0.70,
            0.15, 0.88, 0.45, 0.60, 0.50,
            0.75, 0.10, 0.65, 0.30, 0.55,
            0.40, 0.48, 0.25, 0.85, 0.72,
            0.05, 0.87, 0.62, 0.28, 0.52,
            0.68, 0.38, 0.78, 0.12, 0.58
        ])
    ],
    valid_pathway_mask=[
        torch.ones(30, dtype=torch.bool)
    ]
)


# -----------------------------------
# Test the function
# -----------------------------------

recall = calculate_recall(main_output, sub_output, top_k=20)

print("Recall:", recall)


{0.949999988079071, 0.9200000166893005, 0.8500000238418579, 0.8999999761581421, 0.8799999952316284, 0.8199999928474426, 0.800000011920929, 0.7799999713897705, 0.75, 0.7300000190734863, 0.699999988079071, 0.6800000071525574, 0.6499999761581421, 0.6299999952316284, 0.6000000238418579, 0.5799999833106995, 0.44999998807907104, 0.5, 0.550000011920929, 0.41999998688697815} {0.8999999761581421, 0.8799999952316284, 0.800000011920929, 0.8700000047683716, 0.8500000238418579, 0.7799999713897705, 0.75, 0.7200000286102295, 0.699999988079071, 0.6800000071525574, 0.6499999761581421, 0.6200000047683716, 0.6000000238418579, 0.5799999833106995, 0.550000011920929, 0.5199999809265137, 0.44999998807907104, 0.47999998927116394, 0.5, 0.4000000059604645}
{0.8999999761581421, 0.8799999952316284, 0.7799999713897705, 0.800000011920929, 0.8500000238418579, 0.75, 0.699999988079071, 0.6800000071525574, 0.6499999761581421, 0.6000000238418579, 0.5799999833106995, 0.550000011920929, 0.44999998807907104, 0.5}
Recall: 0

In [8]:
def plot_loss_curves(history, output_path=None):
    """Plot mean training and validation RankNet loss by epoch."""
    import matplotlib.pyplot as plt

    frame = pd.DataFrame(history)
    required = {"epoch", "train_loss", "validation_loss"}
    if not required.issubset(frame.columns):
        raise ValueError(f"history must contain {sorted(required)}")
    figure, axis = plt.subplots(figsize=(7, 4))
    axis.plot(frame["epoch"], frame["train_loss"],
              label="Training RankNet loss")
    axis.plot(frame["epoch"], frame["validation_loss"],
              label="Validation RankNet loss")
    axis.set(xlabel="Epoch", ylabel="Mean pairwise BCE loss",
             title="Pathway-ranking loss")
    axis.grid(alpha=0.25)
    axis.legend()
    figure.tight_layout()
    if output_path is not None:
        figure.savefig(output_path, dpi=150, bbox_inches="tight")
    return figure, axis


In [9]:
def save_test_history(history, output_path="test_history.csv"):
    """Save the sample-level test history with readable pathway lists."""
    frame = pd.DataFrame(history).copy()
    required = {
        "test_loss", "dataset_id", "dataset_name", "top_20_pathways"
    }
    if not required.issubset(frame.columns):
        raise ValueError(f"history must contain {sorted(required)}")
    frame["top_20_pathways"] = frame["top_20_pathways"].apply(
        lambda pathways: "; ".join(map(str, pathways))
    )
    output_path = Path(output_path)
    frame.to_csv(output_path, index=False)
    return output_path


def plot_test_losses(history, output_path=None):
    """Plot the mean test loss for every held-out dataset."""
    import matplotlib.pyplot as plt

    frame = pd.DataFrame(history).copy()
    required = {"test_loss", "dataset_id", "dataset_name"}
    if not required.issubset(frame.columns):
        raise ValueError(f"history must contain {sorted(required)}")
    if frame.empty:
        raise ValueError("test history must not be empty")
    if not np.isfinite(frame["test_loss"].astype(float)).all():
        raise ValueError("test_loss values must be finite")

    frame = frame.groupby(
        ["dataset_id", "dataset_name"], as_index=False, sort=True
    )["test_loss"].mean()
    labels = [
        name if name == dataset_id else f"{name}\n({dataset_id})"
        for dataset_id, name in zip(
            frame["dataset_id"].astype(str),
            frame["dataset_name"].astype(str),
        )
    ]
    figure, axis = plt.subplots(
        figsize=(max(7, 0.9 * len(frame)), 4.5)
    )
    positions = np.arange(len(frame))
    bars = axis.bar(positions, frame["test_loss"].astype(float))
    axis.set(
        xlabel="Test dataset",
        ylabel="Mean RankNet test loss",
        title="Test loss by held-out dataset",
    )
    axis.set_xticks(positions, labels, rotation=45, ha="right")
    axis.grid(axis="y", alpha=0.25)
    axis.bar_label(bars, fmt="%.4f", padding=3, fontsize=8)
    figure.tight_layout()
    if output_path is not None:
        figure.savefig(output_path, dpi=150, bbox_inches="tight")
    return figure, axis


def plot_test_metrics(history, output_path=None):
    """Plot mean loss, recall, and discrimination by gene retention."""
    import matplotlib.pyplot as plt

    frame = pd.DataFrame(history).copy()
    metrics = ["test_loss", "recall", "discrimination"]
    required = set(metrics) | {
        "dataset_id", "dataset_name", "retained_gene_fraction"
    }
    if not required.issubset(frame.columns):
        raise ValueError(f"history must contain {sorted(required)}")
    if frame.empty:
        raise ValueError("test history must not be empty")
    if not np.isfinite(frame[metrics].astype(float).to_numpy()).all():
        raise ValueError("test metric values must be finite")

    summary = frame.groupby(
        ["dataset_id", "dataset_name", "retained_gene_fraction"],
        as_index=False, sort=True,
    )[metrics].mean()
    figure, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    titles = {
        "test_loss": "Mean RankNet test loss",
        "recall": "Top-20 pathway recall",
        "discrimination": "Discrimination success rate",
    }
    for (dataset_id, dataset_name), dataset_frame in summary.groupby(
        ["dataset_id", "dataset_name"], sort=True
    ):
        dataset_frame = dataset_frame.sort_values("retained_gene_fraction")
        label = (str(dataset_name) if str(dataset_name) == str(dataset_id)
                 else f"{dataset_name} ({dataset_id})")
        for axis, metric in zip(axes, metrics):
            axis.plot(
                dataset_frame["retained_gene_fraction"],
                dataset_frame[metric], marker="o", label=label,
            )

    for axis, metric in zip(axes, metrics):
        axis.set(
            xlabel="Retained gene fraction",
            ylabel=titles[metric],
            title=titles[metric],
        )
        axis.grid(alpha=0.25)
        if metric in {"recall", "discrimination"}:
            axis.set_ylim(-0.02, 1.02)
    handles, labels = axes[-1].get_legend_handles_labels()
    figure.legend(
        handles, labels, loc="lower center",
        ncol=min(4, max(1, len(labels))),
    )
    figure.tight_layout(rect=(0, 0.13, 1, 1))
    if output_path is not None:
        figure.savefig(output_path, dpi=150, bbox_inches="tight")
    return figure, axes


## Training from the repository DEG CSV files

This cell loads every `*_DEG_all_genelevel.csv` file in
`micro_gene2_outputs`, builds the real KEGG mask, splits by whole dataset, and
creates `PairSample` lists with `build_pair_samples`. Each CSV contains one
already-computed main DEG table, not independently recalculated sub-dataset
statistics. For a runnable experiment, `build_masked_subdataset_groups` makes
the requested number of reproducible sub-inputs at every configured retained-gene
fraction. Use normalized precomputed main/sub DEG tables for a production study.


In [ ]:
DEG_DIRECTORY = Path("..")
DEG_PATTERN = "*_DEG_all_genelevel.csv"
PATHWAY_FILE = Path("KEGG_2021_Human.gmt")
SUBDATASETS_PER_DATASET = 7  # Number generated for each fraction
RETAINED_GENE_FRACTION = [0.1 ,0.4,0.6]
TOTAL_SUBDATASETS_PER_DATASET = (
    SUBDATASETS_PER_DATASET * len(RETAINED_GENE_FRACTION)
)

deg_files = sorted(DEG_DIRECTORY.glob(DEG_PATTERN))
if len(deg_files) < 3:
    raise FileNotFoundError(
        f"Need at least three DEG CSV files in {DEG_DIRECTORY}; found {len(deg_files)}")
if not PATHWAY_FILE.exists():
    raise FileNotFoundError(PATHWAY_FILE)

def dataset_id_from_deg_path(path):
    return path.stem.removesuffix("_DEG_all_genelevel")

main_deg_tables = {
    dataset_id_from_deg_path(path): load_deg_csv(
        path, dataset_id=dataset_id_from_deg_path(path))
    for path in deg_files
}
real_pathways = load_pathways(PATHWAY_FILE)
all_gene_ids = set()
for table in main_deg_tables.values():
    all_gene_ids.update(table.gene_id.astype(str))
for genes in real_pathways.values():
    all_gene_ids.update(map(str, genes))
gene_vocab = GeneVocabulary.from_genes(all_gene_ids)
pathway_names, pathway_mask = build_pathway_gene_mask(
    real_pathways, gene_vocab)
# pathway_gene_counts = validate_pathway_gene_mask(
#     pathway_mask, pathway_names, gene_vocab)

dataset_groups = build_masked_subdataset_groups(
    main_deg_tables, gene_vocab,
    subdatasets_per_dataset=SUBDATASETS_PER_DATASET,
    retained_gene_fraction=RETAINED_GENE_FRACTION,
    seed=CONFIG.seed,
)
all_main_rows = pd.concat(main_deg_tables.values(), ignore_index=True)
train_ids, validation_ids, test_ids = split_dataset_ids(
    all_main_rows, fractions=(0.70, 0.15, 0.15), seed=CONFIG.seed)
groups_by_id = {group.dataset_id: group for group in dataset_groups}

training_samples = build_pair_samples(
    [groups_by_id[dataset_id] for dataset_id in train_ids],
    subdatasets_per_dataset=TOTAL_SUBDATASETS_PER_DATASET, seed=CONFIG.seed)
validation_samples = build_pair_samples(
    [groups_by_id[dataset_id] for dataset_id in validation_ids],
    subdatasets_per_dataset=TOTAL_SUBDATASETS_PER_DATASET, seed=CONFIG.seed,
    shuffle=False)
test_samples = build_pair_samples(
    [groups_by_id[dataset_id] for dataset_id in test_ids],
    subdatasets_per_dataset=TOTAL_SUBDATASETS_PER_DATASET, seed=CONFIG.seed,
    shuffle=False)
sample_values, sample_available = samples_to_input_matrix(training_samples)

training_config = Config(
    subdataset_mode="gene_row_masking_demo",
    top_pathway_fraction=0.15,
    top_pathway_count=None,
    main_margin=0.1,
    gamma=1.0,
    learning_rate=1e-3,
    weight_decay=1e-4,
    maximum_epochs=60,
)
trained_model = PathwayRankingNetwork(
    len(gene_vocab), pathway_mask, training_config).to(DEVICE)
training_history, optimizer, best_validation_metrics = train_model(
    trained_model, training_samples, validation_samples, training_config)
test_history = test_model(
    trained_model, test_samples, pathway_names, config=training_config)
test_loss = float(test_history["test_loss"].mean())
test_recall = float(test_history["recall"].mean())
test_discrimination = float(test_history["discrimination"].mean())
test_history_file = save_test_history(test_history, "test_history.csv")

print({
    "csv_files": len(deg_files),
    "train_datasets": len(train_ids),
    "validation_datasets": len(validation_ids),
    "test_datasets": len(test_ids),
    "training_pairs": len(training_samples),
    "validation_pairs": len(validation_samples),
    "test_pairs": len(test_samples),
    "genes": len(gene_vocab),
    "pathways": len(pathway_names),
    "pair_matrix_shape": tuple(sample_values.shape),
    "best_validation_loss": best_validation_metrics["validation_loss"],
    "test_loss": test_loss,
    "test_recall": test_recall,
    "test_discrimination": test_discrimination,
})
print(training_history.tail().to_string(index=False))
loss_figure, loss_axis = plot_loss_curves(
    training_history, "ranknet_loss_curve.png")
test_loss_figure, test_loss_axis = plot_test_losses(
    test_history, "test_loss_by_dataset.png")
test_metrics_figure, test_metrics_axes = plot_test_metrics(
    test_history, "test_metrics_by_retained_fraction.png")


KeyboardInterrupt: 